# 02 · Evaluation analysis

Reads the saved results of `python -m plant_rag.evaluate` (no API calls, safe to re-run) and produces the
results table, the charts in `docs/figures/`, and the failure cases.

- `RUN`: the final run on the **test** split (40 questions, never used for tuning).
- `DEV_RUN`: the dev run used to choose setup E and the refusal threshold (shown for comparison only).
- Grades (faithfulness / correctness, 0–2) come from `eval/results/<RUN>/grades.jsonl`, written with
  `eval/faithfulness_rubric.md`.

Setups: A = BM25 only · B = dense only · C = hybrid (RRF) · D = hybrid + reranker · E = D + query translation (default).

In [ ]:
# --- Setup and settings -------------------------------------------------------
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from plant_rag.config import REPO_ROOT

RUN, DEV_RUN = "test_v1", "dev_v2"
SETUPS = ["A", "B", "C", "D", "E"]
SETUP_NAMES = {"A": "A · BM25", "B": "B · Dense", "C": "C · Hybrid", "D": "D · Hybrid+rerank", "E": "E · D+translation"}
FIG_DIR = REPO_ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
N_FAILURES_SHOWN = 6

# Chart style: reference palette (dataviz skill), light surface; slots 1-2 are validated all-pairs.
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834"]            # categorical slot 1 (blue), slot 2 (orange)
ORDINAL = ["#86b6ef", "#3987e5", "#1c5cab"]  # blue ramp steps 250 / 400 / 550 for scores 0 / 1 / 2
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": AXIS,
                     "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})


def load_rows(run: str) -> pd.DataFrame:
    rows = {}
    for line in open(REPO_ROOT / "eval" / "results" / run / "per_question.jsonl", encoding="utf-8"):
        r = json.loads(line)
        rows[(r["qid"], r["setup"])] = r          # last row per (question, setup) wins
    return pd.DataFrame(rows.values())


questions = pd.read_json(REPO_ROOT / "eval" / "questions.jsonl", lines=True).set_index("qid")
rows = load_rows(RUN)
dev_rows = load_rows(DEV_RUN)
grades_path = REPO_ROOT / "eval" / "results" / RUN / "grades.jsonl"
grades = pd.read_json(grades_path, lines=True) if grades_path.exists() else pd.DataFrame()
print(f"{RUN}: {rows.qid.nunique()} questions x {rows.setup.nunique()} setups | grades: {len(grades)}")

## 1. Results table (test split)

Counts are shown with percentages: with 34 answerable test questions, one question moves a percentage by ~3 points.

In [ ]:
def frac(n, d):
    return f"{n}/{d} ({100 * n / d:.0f}%)" if d else "–"


def results_table(rows: pd.DataFrame) -> pd.DataFrame:
    out = []
    for s in SETUPS:
        r = rows[rows.setup == s]
        ans, una = r[r.answerable], r[~r.answerable]
        hit5 = ans.first_gold_rank.fillna(99) <= 5
        cross = ans.cross_lingual
        rec = {"setup": SETUP_NAMES[s],
               "Recall@5": frac(int(hit5.sum()), len(ans)),
               "same-language": frac(int(hit5[~cross].sum()), int((~cross).sum())),
               "cross-language": frac(int(hit5[cross].sum()), int(cross.sum())),
               "MRR@10": round(sum(1 / x for x in ans.first_gold_rank.dropna() if x <= 10) / len(ans), 3),
               "median search s": round(r.search_s.median(), 1)}
        if "refused" in r and r.refused.notna().any():
            given = ans[~ans.refused.astype(bool)]
            rec |= {"correct refusals": frac(int(una.refused.astype(bool).sum()), len(una)),
                    "false refusals": frac(int(ans.refused.astype(bool).sum()), len(ans)),
                    "answers citing gold": frac(int((given.cited_gold > 0).sum()), len(given)),
                    "citation precision": frac(int(given.cited_gold.sum()), int(given.cited.map(len).sum())),
                    "median LLM s": round(r.llm_s.median(), 1)}
        out.append(rec)
    return pd.DataFrame(out).set_index("setup")


table = results_table(rows)
table.to_csv(REPO_ROOT / "eval" / "results" / RUN / "results_table.csv", encoding="utf-8-sig")
table

In [ ]:
# Dev split, for comparison only (setup E and the refusal threshold were chosen on these 20 questions)
results_table(dev_rows)

## 2. Retrieval: is the gold clause in the top 5?

In [ ]:
def recall_split(rows):
    vals = {}
    for s in SETUPS:
        ans = rows[(rows.setup == s) & rows.answerable]
        hit = ans.first_gold_rank.fillna(99) <= 5
        vals[s] = {"same": (int(hit[~ans.cross_lingual].sum()), int((~ans.cross_lingual).sum())),
                   "cross": (int(hit[ans.cross_lingual].sum()), int(ans.cross_lingual.sum()))}
    return vals


vals = recall_split(rows)
fig, ax = plt.subplots(figsize=(8, 3.8))
width, xs = 0.38, range(len(SETUPS))
for k, (key, label) in enumerate([("same", "Same-language questions"), ("cross", "Cross-language questions")]):
    for i, s in enumerate(SETUPS):
        n, d = vals[s][key]
        x = i + (k - 0.5) * width
        ax.bar(x, 100 * n / d, width, color=SERIES[k], edgecolor=SURFACE, linewidth=2,
               label=label if i == 0 else None)
        ax.text(x, 100 * n / d + 2, f"{n}/{d}", ha="center", va="bottom", fontsize=8, color=INK2)
ax.set_xticks(list(xs), [SETUP_NAMES[s] for s in SETUPS])
ax.set_ylim(0, 112)
ax.set_ylabel("Recall@5 (%)")
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_title("Gold clause in the top 5 (test split)", loc="left", color=INK, pad=28)
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncols=2, fontsize=9, borderaxespad=0.2)
plt.tight_layout()
fig.savefig(FIG_DIR / "recall_by_setup.png", dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
mrr = [sum(1 / x for x in rows[(rows.setup == s) & rows.answerable].first_gold_rank.dropna() if x <= 10)
       / rows[(rows.setup == s) & rows.answerable].shape[0] for s in SETUPS]
ax.bar(range(len(SETUPS)), mrr, 0.55, color=SERIES[0], edgecolor=SURFACE, linewidth=2)
for i, v in enumerate(mrr):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8, color=INK2)
ax.set_xticks(range(len(SETUPS)), [SETUP_NAMES[s] for s in SETUPS])
ax.set_ylim(0, 1.1)
ax.set_ylabel("MRR@10")
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_title("How high the first gold clause ranks (test split, 1.0 = always first)", loc="left", color=INK)
plt.tight_layout()
fig.savefig(FIG_DIR / "mrr_by_setup.png", dpi=200)
plt.show()

## 3. Answers: faithfulness and correctness (graded, setups C / D / E)

Scores from `eval/faithfulness_rubric.md`: **faithfulness** = supported by the evidence shown to the model;
**correctness** = matches the gold answer. Refused answers are not graded (see the refusal columns above).

In [ ]:
if grades.empty:
    print("No grades yet: grade with eval/faithfulness_rubric.md into", grades_path)
else:
    summary = (grades.groupby("setup")[["faithfulness", "correctness"]]
               .agg(lambda s: f"{(s == 2).sum()}/{len(s)} full · {(s == 1).sum()} partial · {(s == 0).sum()} zero"))
    display(summary)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
    for ax, metric in zip(axes, ["faithfulness", "correctness"]):
        setups = sorted(grades.setup.unique())
        left = [0] * len(setups)
        for score, color, label in zip([2, 1, 0], ORDINAL[::-1], ["2 · full", "1 · partial", "0 · none"]):
            counts = [int(((grades.setup == s) & (grades[metric] == score)).sum()) for s in setups]
            ax.barh([SETUP_NAMES[s] for s in setups], counts, left=left, color=color, edgecolor=SURFACE,
                    linewidth=2, label=label)
            for y, (l, c) in enumerate(zip(left, counts)):
                if c:
                    ax.text(l + c / 2, y, str(c), ha="center", va="center", fontsize=8,
                            color="#ffffff" if score > 0 else INK)
            left = [a + b for a, b in zip(left, counts)]
        ax.set_title(metric.capitalize(), loc="left", color=INK)
        ax.xaxis.grid(True, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
        ax.set_xlabel("answers")
    axes[0].legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.08), ncols=3, fontsize=9)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "answer_quality.png", dpi=200)
    plt.show()

## 4. Latency on the laptop (median seconds per question)

Search = CPU work + query embedding (+ translation for E); LLM = Gemini answer time. The reranker
dominates D and E; first-call model loading (~10 s) is excluded by using the median. A and B produce no LLM
answers (retrieval only). Setups run in order A→E, so C–E reuse the question embeddings cached by B: their
search times exclude the ~0.4 s embedding call (E's translated question is embedded fresh).

In [ ]:
lat = pd.DataFrame({s: {"search": rows[rows.setup == s].search_s.median(),
                        "LLM": rows[rows.setup == s].llm_s.median() if "llm_s" in rows else float("nan")}
                    for s in SETUPS}).T.fillna(0)
fig, ax = plt.subplots(figsize=(8, 3))
names = [SETUP_NAMES[s] for s in SETUPS]
ax.barh(names, lat["search"], color=SERIES[0], edgecolor=SURFACE, linewidth=2, label="Search")
ax.barh(names, lat["LLM"], left=lat["search"], color=SERIES[1], edgecolor=SURFACE, linewidth=2, label="LLM answer")
for y, (srch, llm) in enumerate(zip(lat["search"], lat["LLM"])):
    ax.text(srch + llm + 0.3, y, f"{srch + llm:.1f} s", va="center", fontsize=8, color=INK2)
ax.invert_yaxis()
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_xlabel("seconds (median)")
ax.set_title("Time per question on the laptop CPU (test split)", loc="left", color=INK, pad=28)
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncols=2, fontsize=9, borderaxespad=0.2)
ax.set_xlim(0, (lat["search"] + lat["LLM"]).max() * 1.12)
plt.tight_layout()
fig.savefig(FIG_DIR / "latency.png", dpi=200)
plt.show()
lat.round(1)

## 5. Failure cases (setup E)

Every test question where the default setup missed the gold clause in the top 5, refused an answerable
question, answered an unanswerable one, or scored below 2 on correctness or faithfulness.

In [ ]:
chunks = {c["chunk_id"]: c for c in map(json.loads, open(REPO_ROOT / "data" / "processed" / "chunks.jsonl", encoding="utf-8"))}
e = rows[rows.setup == "E"].set_index("qid")
g = grades[grades.setup == "E"].set_index("qid") if not grades.empty else pd.DataFrame()
fails = []
for qid, r in e.iterrows():
    reasons = []
    if r.answerable and not (r.first_gold_rank and r.first_gold_rank <= 5):
        reasons.append(f"gold not in top 5 (rank {r.first_gold_rank})")
    if r.answerable and r.refused:
        reasons.append("false refusal")
    if not r.answerable and not r.refused:
        reasons.append("answered an unanswerable question")
    if qid in g.index:
        if g.loc[qid, "correctness"] < 2:
            reasons.append(f"correctness {g.loc[qid, 'correctness']}")
        if g.loc[qid, "faithfulness"] < 2:
            reasons.append(f"faithfulness {g.loc[qid, 'faithfulness']}")
    if reasons:
        fails.append((qid, reasons))
print(f"{len(fails)} of {len(e)} test questions have an issue in setup E\n")
for qid, reasons in fails[:N_FAILURES_SHOWN]:
    q, r = questions.loc[qid], e.loc[qid]
    print(f"### {qid} [{q.type}, {q.lang}->{q.doc_lang or '-'}] {'; '.join(reasons)}")
    print("Q:", q.question)
    print("Gold:", q.gold_answer)
    print("Answer:", r.answer[:500] if isinstance(r.answer, str) else r.answer)
    print("Top 3 retrieved:", [f"{chunks[i]['doc_id']} {chunks[i]['clause_id']}" for i in r.retrieved[:3]])
    if qid in g.index:
        print("Grader:", g.loc[qid, "reason"])
    print()

## Findings (test split, 2026-10-13)

**Result sentence.** On 40 held-out test questions, hybrid search + reranker + query translation (setup E)
found the gold clause in the top 5 for **34/34** answerable questions (BM25 alone: 18/34; hybrid: 23/34),
including **16/16** cross-language questions, and refused **6/6** unanswerable questions with **1/34** false refusal.
All 33 answers it gave were fully faithful to their evidence; 32/33 fully correct (grader).

1. **Cross-language is the hard part, and translation fixes it.** Same-language questions are easy for every
   setup. All misses in C and D (and both misses of dense-only B) are cross-language questions: BM25 only
   matches the question's language, so fusion buries the right passage, and the reranker scores
   Korean–English pairs too low. Translating the question (E) lifts cross-language Recall@5 from 5/16 (C) and
   12/16 (D) to 16/16.
2. **Dense-only search (B) is a strong, fast baseline**: 32/34 Recall@5 with near-zero search time, versus ~18 s
   for E on the laptop CPU (mostly the reranker). E wins on ranking (MRR 0.92 vs 0.86), refusals, and on the
   cross-language questions B still missed.
3. **The model rarely invents.** Faithfulness was 2/2 for all 87 graded answers (C, D, E). Errors are omissions
   (a missing condition or time limit), not made-up facts.
4. **Refusal threshold is a cost filter, not a perfect separator.** On test, unanswerable questions reached
   reranker scores up to 0.56 (refused correctly by the model), and one answerable question scored 0.065
   (q030: the question says "노드", the guide says "검토구간") and was refused before the API call. The threshold
   (0.15) was tuned on dev and deliberately not re-tuned on test.

**Three failure cases**
- **q030 (E, false refusal).** HAZOP node question in Korean; the gold clause ranked #1 but the reranker scored it
  0.065 because of different terminology (노드 vs 검토구간). Lesson: a synonym gap between users and the standard;
  fixes would be a lower threshold (more API calls) or query expansion with standard terminology.
- **q010 (C, cut-off answer).** US MOC question in Korean; hybrid search (no translation) retrieved only KOSHA
  passages, and the answer was truncated because Gemini's "thinking" used ~1,960 of the 2,048 output tokens.
  Fixed after the run (`max_output_tokens: 8192` + a warning on truncation); setup E was not affected.
- **q006 (E, partially correct).** The 10,000 lb threshold is right, but the answer leans on the 2000 booklet
  wording and drops the 2025 CFR qualifier (Category 1 gases / flashpoint < 100 °F). Lesson: when sources of
  different editions disagree, the answer should prefer the newest one; the prompt does not say so yet.

**Caveats.** Small set (40 test questions; one question ≈ 3 points). Gold labels and grades are model-made
(Claude), with a 17-answer spot-check by a second Claude session (17/17 agreement, not fully blind) instead of a
human. Latency is from the laptop CPU; the cache means re-runs are faster than first runs.